# Multi-Embodiment Experiment: H1 to G1 Stand Task

This notebook demonstrates transfer learning across different robot embodiments.
We load a model trained on h1-stand-v0 and evaluate it on g1-stand-v0.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MUJOCO_GL"] = "egl"


import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
)

from fast_td3 import Critic
from fast_td3.actors import ActorEGNN, Actor

## Configuration

We configure the environment for the **target robot (G1)** but will load a model trained on **H1**.

In [ ]:
from fast_td3.hyperparams import HumanoidBenchArgs

# Source robot (model trained on this)
source_robot = "h1"
source_task = "stand-v0"

# Target robot (evaluate model on this)
target_robot = "g1"
target_task = "stand-v0"

# Create args for the TARGET environment (g1-stand-v0)
args = HumanoidBenchArgs(
    env_name=f"{target_robot}-{target_task}",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192,
    actor_hidden_dim=384,
)

print(f"Evaluating model trained on {source_robot}-{source_task} on environment {target_robot}-{target_task}")

In [ ]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

## Environment Setup

Create the G1 stand environment for evaluation.

In [ ]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
eval_envs = HumanoidBenchEnv(args.env_name, args.num_envs, device=device)
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = eval_envs.num_actions
n_obs = eval_envs.num_obs if type(eval_envs.num_obs) == int else eval_envs.num_obs[0]
if eval_envs.asymmetric_obs:
    n_critic_obs = (
        eval_envs.num_privileged_obs
        if type(eval_envs.num_privileged_obs) == int
        else eval_envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

print(f"Target environment: {args.env_name}")
print(f"Observation space: {n_obs}")
print(f"Action space: {n_act}")

## Model Loading

Load the EGNN actor trained on H1-stand-v0.

**Note:** You'll need to provide the path to your trained H1 stand model checkpoint.
Replace the checkpoint_path below with your actual model path.

In [ ]:
# TODO: Update this path to your actual H1 stand model checkpoint
checkpoint_path = "./models/egnn_h1-stand-v0_16envs_150001steps_xxxxxx_final.pt"

# If you don't have a trained model yet, you can create a placeholder:
# For demonstration purposes, we'll create the actor architecture
# but you'll need an actual trained checkpoint to get meaningful results

obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
xanchor_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)

# Actor setup - using SOURCE robot (h1) architecture
# but it will receive observations from TARGET robot (g1)
actor = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=64,
    n_layers=4,
    act_fn="relu",
    robot=source_robot,  # Use source robot (h1) for the graph structure
    env_name=f"{source_robot}_{source_task.replace('-', '_')}",
)

# Load checkpoint if it exists
import os
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from: {checkpoint_path}")
    torch_checkpoint = torch.load(
        f"{checkpoint_path}", map_location=device, weights_only=False
    )
    obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
    xanchor_normalizer.load_state_dict(torch_checkpoint["xanchor_normalizer_state"])
    actor.load_state_dict(torch_checkpoint["actor_state_dict"])
    print("Checkpoint loaded successfully!")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")
    print("Using randomly initialized weights for demonstration.")
    print("To get meaningful results, train a model on h1-stand-v0 first.")

normalize_obs = obs_normalizer.forward
normalize_xanchor = xanchor_normalizer.forward

print(f"Actor parameters: {sum(p.numel() for p in actor.parameters())}")

## Evaluation Functions

In [ ]:
def evaluate():
    """Evaluate the H1 model on the G1 environment."""
    obs_normalizer.eval()
    xanchor_normalizer.eval()
    num_eval_envs = eval_envs.num_envs
    episode_returns = torch.zeros(num_eval_envs, device=device)
    episode_lengths = torch.zeros(num_eval_envs, device=device)
    done_masks = torch.zeros(num_eval_envs, dtype=torch.bool, device=device)
    
    obs, xanchor = eval_envs.reset()
    for _ in range(eval_envs.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):  
            obs = normalize_obs(obs)
            xanchor = normalize_xanchor(xanchor)
            actions = actor(obs, xanchor)

        next_obs, rewards, dones, _ , next_xanchor = eval_envs.step(actions.float())
        episode_returns = torch.where(
            ~done_masks, episode_returns + rewards, episode_returns
        )
        episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
        done_masks = torch.logical_or(done_masks, dones)
        if done_masks.all():
            break
        obs = next_obs
        xanchor = next_xanchor

    obs_normalizer.train()
    xanchor_normalizer.train()
    return episode_returns.mean().item(), episode_lengths.mean().item()

In [ ]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
    """
    Convert a list of numpy arrays to an HTML5 video element.

    Args:
        frames (list): List of numpy arrays representing video frames
        fps (int): Frames per second for the video

    Returns:
        HTML object containing the video element
    """
    # Create a temporary file to store the video
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
        temp_filename = temp_file.name

    # Save frames as video
    imageio.mimsave(temp_filename, frames, fps=fps)

    # Read the video file and encode it to base64
    with open(temp_filename, "rb") as f:
        video_data = f.read()
    video_b64 = base64.b64encode(video_data).decode("utf-8")

    # Create HTML video element
    video_html = f"""
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """

    # Clean up the temporary file
    os.unlink(temp_filename)

    return HTML(video_html)


def render_with_rollout():
    """Render a rollout of the H1 model on the G1 environment."""
    obs_normalizer.eval()
    xanchor_normalizer.eval()

    # Quick rollout for rendering
    if env_type == "humanoid_bench":
        obs, xanchor = render_env.reset()
        renders = [render_env.render()]

    for i in range(render_env.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):
            obs = normalize_obs(obs)
            xanchor = normalize_xanchor(xanchor)
            actions = actor(obs, xanchor)
        next_obs, _, done, _, next_xanchor = render_env.step(actions.float())
        if i % 2 == 0:
            if env_type == "humanoid_bench":
                renders.append(render_env.render())
        if done.any():
            break
        obs = next_obs
        xanchor = next_xanchor

    obs_normalizer.train()
    xanchor_normalizer.train()
    video_html = frames_to_video_html(renders, fps=30)
    display(video_html)


## Run Evaluation

Evaluate the H1-trained model on the G1 environment.

In [ ]:
mean_return, mean_length = evaluate()
print(f"\n=== Multi-Embodiment Evaluation Results ===")
print(f"Source: {source_robot}-{source_task}")
print(f"Target: {target_robot}-{target_task}")
print(f"Mean Episode Return: {mean_return:.2f}")
print(f"Mean Episode Length: {mean_length:.2f}")
print("=========================================\n")

## Visualize Performance

Render a video showing how the H1 model performs on the G1 robot.

In [ ]:
render_with_rollout()

## Analysis Notes

This notebook demonstrates multi-embodiment transfer learning:

1. **Source Robot**: H1 (19 joints)
2. **Target Robot**: G1 (37 joints - includes hand joints)
3. **Task**: Stand (maintain upright posture)

The key insight is that the EGNN architecture can generalize across different robot morphologies
because it operates on the graph structure of the robot rather than fixed feature vectors.

### Expected Behavior:
- The model may perform reasonably well if the core standing behavior transfers
- Hand joints in G1 may not be well-controlled since they don't exist in H1
- The shared structure (legs, torso, arms without hands) should have better transfer

### To Improve Transfer:
1. Train on both embodiments simultaneously
2. Use domain randomization during training
3. Fine-tune the transferred model on the target robot
4. Use a shared embedding space for similar joint types